# 13 - Interpretabilita con SHAP
Analisi di Explainable AI (XAI) con SHAP per i modelli XGBoost e GRU, a livello globale e locale (Sezione 4.3 della tesi).

In [ ]:
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/t1dbg'
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

In [ ]:
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from lib.data import load_splits, rescale_data, categorize_glucose, L_BOUND, U_BOUND
from lib.tf_dnn import create_gru_model

warnings.filterwarnings("ignore")
shap.initjs()

## Configurazione

In [ ]:
config = {
    "xgb_model_path": "models/test_set/xgb.pickle",
    "gru_model_path": "models/val_set/gru.weights.h5",
    "output_dir": "xai/shap_analysis",
    "plots_dir": "plots/shap_analysis",
    "background_size": 50,
    "explanation_size": 250,
    "random_seed": 42,
}

RANDOM_SEED = config["random_seed"]
np.random.seed(RANDOM_SEED)

os.makedirs(config["output_dir"], exist_ok=True)
os.makedirs(config["plots_dir"], exist_ok=True)

## Caricamento dati

In [ ]:
train_set, val_set, test_set, X_cols, y_cols = load_splits()

print(f"Train set: {len(train_set)} samples")
print(f"Val set:   {len(val_set)} samples")
print(f"Test set:  {len(test_set)} samples")
print(f"Features:  {X_cols}")
print(f"Target:    {y_cols}")

## Campionamento condiviso

Selezioniamo un insieme comune di campioni di background (dal train set) e di spiegazione (dal train set) da usare per entrambi i modelli, in modo da avere risultati confrontabili.

In [ ]:
# Background samples (dal training set)
background_indices = np.random.choice(
    len(train_set), size=config["background_size"], replace=False
)
background_data = train_set.iloc[background_indices].copy()

print("Background samples per classe:")
print(background_data["bgClass"].value_counts())

# Explanation samples (dal training set)
explanation_indices = np.random.choice(
    len(train_set), size=config["explanation_size"], replace=False
)
explanation_data = train_set.iloc[explanation_indices].copy()

print("\nExplanation samples per classe:")
print(explanation_data["bgClass"].value_counts())

In [ ]:
# Identificazione dei casi clinici rappresentativi (ipo, normo, iper)
clinical_cases = {}

for bg_class in ["Hypo", "Normal", "Hyper"]:
    class_data = explanation_data[explanation_data["bgClass"] == bg_class]
    if len(class_data) > 0:
        median_target = class_data["lead30"].median()
        distances = np.abs(class_data["lead30"] - median_target)
        best_idx = distances.idxmin()
        clinical_cases[bg_class] = {
            "index": best_idx,
            "data": class_data.loc[best_idx],
            "target_value": class_data.loc[best_idx, "lead30"],
        }
        print(f"{bg_class}: Target = {clinical_cases[bg_class]['target_value']:.4f}")
    else:
        print(f"Warning: nessun campione {bg_class} trovato")

In [ ]:
# Preparazione matrici di feature per SHAP (2D)
X_background = background_data[X_cols].values
X_explanation = explanation_data[X_cols].values

print(f"Background data shape: {X_background.shape}")
print(f"Explanation data shape: {X_explanation.shape}")

## SHAP per XGBoost (TreeExplainer)

In [ ]:
# Caricamento modello XGBoost
with open(config["xgb_model_path"], "rb") as f:
    xgb_model = pickle.load(f)

print(f"Modello XGBoost caricato da {config['xgb_model_path']}")
print(f"Tipo: {type(xgb_model)}")
if hasattr(xgb_model, "n_estimators"):
    print(f"Numero di estimatori: {xgb_model.n_estimators}")
if hasattr(xgb_model, "max_depth"):
    print(f"Max depth: {xgb_model.max_depth}")

In [ ]:
# TreeExplainer - ottimale per modelli basati su alberi
xgb_explainer = shap.TreeExplainer(xgb_model, X_background)

print("Calcolo dei valori SHAP per XGBoost...")
xgb_shap_values = xgb_explainer.shap_values(X_explanation)

print(f"SHAP values shape: {xgb_shap_values.shape}")
print(f"Expected value (baseline): {xgb_explainer.expected_value:.4f}")

<cell_type>markdown</cell_type>## SHAP per GRU (KernelExplainer)

Il modello GRU utilizza TensorFlow/Keras (`lib.tf_dnn.create_gru_model`). Poiché SHAP non supporta direttamente i modelli RNN, usiamo `KernelExplainer` con una funzione wrapper che gestisce il reshape da 2D a 3D.

In [ ]:
# Caricamento modello GRU (TensorFlow/Keras)
gru_model = create_gru_model()
gru_model.load_weights(config["gru_model_path"])

print(f"Modello GRU caricato da {config['gru_model_path']}")
print(f"Parametri totali: {gru_model.count_params():,}")

In [ ]:
def gru_predict_wrapper(X):
    """Wrapper per il modello GRU: accetta input 2D e restituisce predizioni 1D."""
    if X.ndim == 2:
        X = X.reshape(X.shape[0], X.shape[1], 1)
    return gru_model.predict(X, verbose=0).flatten()

In [ ]:
# KernelExplainer - model-agnostic, necessario per modelli non-tree
print("Creazione KernelExplainer per GRU...")
gru_explainer = shap.KernelExplainer(gru_predict_wrapper, X_background)

print("Calcolo dei valori SHAP per GRU (puo richiedere diversi minuti)...")
gru_shap_values = gru_explainer.shap_values(X_explanation, nsamples=512)

print(f"SHAP values shape: {gru_shap_values.shape}")
print(f"Expected value (baseline): {gru_explainer.expected_value:.4f}")

## Rescaling dei valori SHAP

I dati sono normalizzati in [-1, 1]. Per l'interpretazione clinica, riscaliamo i valori SHAP e le feature nell'intervallo originale in mg/dL.

In [ ]:
scaling_factor = (U_BOUND - L_BOUND) / 2
print(f"Fattore di scala per SHAP values: {scaling_factor}")

# Rescale delle feature
explanation_features_rescaled = explanation_data[X_cols].copy()
for col in X_cols:
    explanation_features_rescaled[col] = (
        (explanation_features_rescaled[col] + 1) * (U_BOUND - L_BOUND) / 2
    ) + L_BOUND

# Rescale dei valori SHAP
xgb_shap_rescaled = xgb_shap_values * scaling_factor
gru_shap_rescaled = gru_shap_values * scaling_factor

# Rescale degli expected values
xgb_expected_rescaled = ((xgb_explainer.expected_value + 1) * (U_BOUND - L_BOUND) / 2) + L_BOUND
gru_expected_rescaled = ((gru_explainer.expected_value + 1) * (U_BOUND - L_BOUND) / 2) + L_BOUND

print(f"\nExpected values riscalati:")
print(f"  XGBoost: {xgb_explainer.expected_value:.4f} -> {xgb_expected_rescaled:.1f} mg/dL")
print(f"  GRU:     {gru_explainer.expected_value:.4f} -> {gru_expected_rescaled:.1f} mg/dL")

## Analisi globale

### Mean SHAP values - Confronto importanza delle feature

In [ ]:
# Importanza media assoluta per ogni feature
xgb_importance = np.abs(xgb_shap_rescaled).mean(axis=0)
gru_importance = np.abs(gru_shap_rescaled).mean(axis=0)

importance_df = pd.DataFrame({
    "Feature": X_cols,
    "XGBoost": xgb_importance,
    "GRU": gru_importance,
})

# Bar plot confronto
fig, ax = plt.subplots(figsize=(12, 8))
x_pos = np.arange(len(X_cols))
width = 0.35

ax.bar(x_pos - width / 2, importance_df["XGBoost"], width, label="XGBoost", alpha=0.8, color="skyblue")
ax.bar(x_pos + width / 2, importance_df["GRU"], width, label="GRU", alpha=0.8, color="lightcoral")

ax.set_xlabel("Features")
ax.set_ylabel("Mean |SHAP Value| (mg/dL)")
ax.set_title("Feature Importance Comparison: XGBoost vs GRU\n(Impact on glucose prediction in mg/dL)")
ax.set_xticks(x_pos)
ax.set_xticklabels(X_cols, rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{config['plots_dir']}/feature_importance_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

importance_df.to_csv(f"{config['plots_dir']}/feature_importance_comparison.csv", index=False)

print("\nTop 3 features per importanza:")
print("XGBoost:")
for i, (_, row) in enumerate(importance_df.nlargest(3, "XGBoost").iterrows()):
    print(f"  {i+1}. {row['Feature']}: {row['XGBoost']:.2f} mg/dL")
print("GRU:")
for i, (_, row) in enumerate(importance_df.nlargest(3, "GRU").iterrows()):
    print(f"  {i+1}. {row['Feature']}: {row['GRU']:.2f} mg/dL")

### Summary plots

In [ ]:
# XGBoost Summary Plot
plt.figure(figsize=(12, 8))
shap.summary_plot(
    xgb_shap_rescaled,
    explanation_features_rescaled.values,
    feature_names=X_cols,
    show=False,
)
plt.title("XGBoost - SHAP Summary Plot (Rescaled Values)", fontsize=14, pad=20)
plt.xlabel("SHAP value (impact on model output in mg/dL)", fontsize=12)
plt.tight_layout()
plt.savefig(f"{config['plots_dir']}/xgb_summary_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# GRU Summary Plot
plt.figure(figsize=(12, 8))
shap.summary_plot(
    gru_shap_rescaled,
    explanation_features_rescaled.values,
    feature_names=X_cols,
    show=False,
)
plt.title("GRU - SHAP Summary Plot (Rescaled Values)", fontsize=14, pad=20)
plt.xlabel("SHAP value (impact on model output in mg/dL)", fontsize=12)
plt.tight_layout()
plt.savefig(f"{config['plots_dir']}/gru_summary_plot.png", dpi=300, bbox_inches="tight")
plt.show()

## Analisi locale

Waterfall plots per i tre casi clinici rappresentativi: ipoglicemia, normoglicemia e iperglicemia.

In [ ]:
# Trova gli indici dei casi clinici nella matrice di spiegazione
clinical_exp_indices = {}
explanation_features_arr = explanation_features_rescaled.values

for bg_class, case_info in clinical_cases.items():
    original_idx = case_info["index"]
    explanation_mask = explanation_data.index == original_idx
    if explanation_mask.any():
        exp_idx = np.where(explanation_mask)[0][0]
        clinical_exp_indices[bg_class] = exp_idx
        target_rescaled = ((case_info["target_value"] + 1) * (U_BOUND - L_BOUND) / 2) + L_BOUND
        print(f"{bg_class}: explanation index {exp_idx}, target = {target_rescaled:.1f} mg/dL")
    else:
        print(f"Warning: caso {bg_class} non trovato nei dati di spiegazione")

In [ ]:
# Waterfall plots per ogni caso clinico
for bg_class, exp_idx in clinical_exp_indices.items():
    print(f"\n--- {bg_class} ---")

    # XGBoost waterfall
    if exp_idx < len(xgb_shap_rescaled):
        plt.figure(figsize=(10, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values=xgb_shap_rescaled[exp_idx],
                base_values=xgb_expected_rescaled,
                data=explanation_features_arr[exp_idx],
                feature_names=X_cols,
            ),
            show=False,
        )
        plt.title(f"XGBoost - {bg_class} Case Waterfall Plot\n(Values in mg/dL)", fontsize=12)
        plt.tight_layout()
        plt.savefig(
            f"{config['plots_dir']}/xgb_waterfall_{bg_class.lower()}.png",
            dpi=300, bbox_inches="tight",
        )
        plt.show()

    # GRU waterfall
    if exp_idx < len(gru_shap_rescaled):
        plt.figure(figsize=(10, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values=gru_shap_rescaled[exp_idx],
                base_values=gru_expected_rescaled,
                data=explanation_features_arr[exp_idx],
                feature_names=X_cols,
            ),
            show=False,
        )
        plt.title(f"GRU - {bg_class} Case Waterfall Plot\n(Values in mg/dL)", fontsize=12)
        plt.tight_layout()
        plt.savefig(
            f"{config['plots_dir']}/gru_waterfall_{bg_class.lower()}.png",
            dpi=300, bbox_inches="tight",
        )
        plt.show()

## Salvataggio risultati

In [ ]:
output_dir = config["output_dir"]

# Salvataggio SHAP values come numpy arrays
np.save(f"{output_dir}/xgb_shap_values.npy", xgb_shap_values)
np.save(f"{output_dir}/gru_shap_values.npy", gru_shap_values)

# Salvataggio feature di spiegazione
explanation_features_df = pd.DataFrame(X_explanation, columns=X_cols)
explanation_features_df.to_csv(f"{output_dir}/explanation_features.csv", index=False)

# Salvataggio feature di background
background_features_df = pd.DataFrame(X_background, columns=X_cols)
background_features_df.to_csv(f"{output_dir}/background_features.csv", index=False)

# Salvataggio expected values
expected_values = {
    "xgb_expected_value": float(xgb_explainer.expected_value),
    "gru_expected_value": float(gru_explainer.expected_value),
}
with open(f"{output_dir}/expected_values.json", "w") as f:
    json.dump(expected_values, f, indent=2)

# Salvataggio stato del campionamento
manager_state = {
    "background_indices": background_indices.tolist(),
    "explanation_indices": explanation_indices.tolist(),
    "clinical_cases": {
        bg_class: {
            "index": int(info["index"]),
            "target_value": float(info["target_value"]),
        }
        for bg_class, info in clinical_cases.items()
    },
}
with open(f"{output_dir}/sampling_manager_state.json", "w") as f:
    json.dump(manager_state, f, indent=2)

# Salvataggio summary complessivo
summary = {
    "analysis_config": config,
    "X_cols": X_cols,
    "background_samples": len(background_data),
    "explanation_samples": len(explanation_data),
    "background_class_distribution": background_data["bgClass"].value_counts().to_dict(),
    "explanation_class_distribution": explanation_data["bgClass"].value_counts().to_dict(),
    "xgb_analysis": {
        "shap_values_shape": list(xgb_shap_values.shape),
        "expected_value": float(xgb_explainer.expected_value),
        "explainer_type": "TreeExplainer",
    },
    "gru_analysis": {
        "shap_values_shape": list(gru_shap_values.shape),
        "expected_value": float(gru_explainer.expected_value),
        "explainer_type": "KernelExplainer",
    },
    "clinical_cases": {
        bg_class: {
            "index": int(info["index"]),
            "target_value": float(info["target_value"]),
        }
        for bg_class, info in clinical_cases.items()
    },
}
with open(f"{output_dir}/analysis_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Risultati salvati in {output_dir}/")
print(f"  - xgb_shap_values.npy")
print(f"  - gru_shap_values.npy")
print(f"  - explanation_features.csv")
print(f"  - background_features.csv")
print(f"  - expected_values.json")
print(f"  - sampling_manager_state.json")
print(f"  - analysis_summary.json")
print(f"\nPlots salvati in {config['plots_dir']}/")